# Sesión 10 — Modelos Generativos: GMM y el Algoritmo EM
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo III · Modelos Generativos Clásicos**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Derivar el modelo generativo de una mezcla de gaussianas (GMM) y su función de verosimilitud.
2. Implementar el algoritmo EM desde cero para ajustar un GMM, con aritmética en espacio log para estabilidad numérica.
3. Contrastar las asignaciones blandas (GMM) con las duras (k-means) y entender cuándo difieren.
4. Seleccionar el número de componentes K mediante BIC, AIC e índice de silueta.
5. Aplicar el GMM como estimador de densidad para la detección de anomalías en señales de ECG.
6. Reconocer el Naive Bayes gaussiano, el LDA y el QDA como casos especiales de GMM.

## Conjunto de datos principal

**Estados de vigilancia EEG** — segmentos de EEG clasificados en tres estados:
vigilia (W), somnolencia (N1/N2), sueño profundo (N3).
Distribución simulada a partir de las estadísticas de potencia espectral reportadas en:

Berry, R.B. et al. (2015). *AASM Scoring Manual* (version 2.2). American Academy of Sleep Medicine.
https://aasm.org/clinical-resources/scoring-manual/

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Bishop, C.M. (2006). *PRML*. §9.1–9.3 (GMM y EM). Springer. |
| ★★★ | Dempster, A.P., Laird, N.M. & Rubin, D.B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *JRSS-B*, 39(1), 1–38. |
| ★★☆ | McLachlan, G.J. & Peel, D. (2000). *Finite Mixture Models*. Wiley. Cap. 1–2. |
| ★★☆ | Izenman, A.J. (2008). *Modern Multivariate Statistical Techniques*. Springer. §14 (Clustering). |
| ★☆☆ | Reynolds, D.A. (2009). Gaussian Mixture Models. *Encyclopedia of Biometrics*. Springer. |

## Parte 0 — Configuración y datos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy import stats
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

# ── Dataset EEG — estados de vigilancia ───────────────────────────────────────
# Características: potencia en 4 bandas (δ, θ, α, β) × 1 canal = 4 dimensiones
# Distribución calibrada con valores de Berry et al. (2015) AASM Scoring Manual
# https://aasm.org/clinical-resources/scoring-manual/

N_por_clase = 200
nombres_estados = ['Vigilia (W)', 'Somnolencia (N1/N2)', 'Sueño profundo (N3)']
nombres_bandas  = ['Delta (0.5–4 Hz)', 'Theta (4–8 Hz)', 'Alpha (8–13 Hz)', 'Beta (13–30 Hz)']

# Medias de potencia relativa por estado (log-espacio, dB)
# W: beta y alpha dominantes, delta bajo
# N1/N2: theta y alpha moderados
# N3: delta dominante (sueño de ondas lentas)
medias_eeg = np.array([
    [0.5,  0.8,  2.0,  1.8],  # Vigilia (W)
    [1.0,  1.8,  1.2,  0.9],  # Somnolencia (N1/N2)
    [3.5,  1.0,  0.4,  0.3],  # Sueño profundo (N3)
])
covs_eeg = [
    np.diag([0.5, 0.6, 0.7, 0.4]),
    np.diag([0.6, 0.7, 0.6, 0.5]),
    np.diag([0.8, 0.5, 0.4, 0.3]),
]

X_eeg_list, y_eeg_list = [], []
for k in range(3):
    Xk = rng.multivariate_normal(medias_eeg[k], covs_eeg[k], N_por_clase)
    X_eeg_list.append(Xk)
    y_eeg_list.append(np.full(N_por_clase, k))

X_eeg = np.vstack(X_eeg_list).astype(np.float32)
y_eeg = np.concatenate(y_eeg_list).astype(int)
N_eeg = len(X_eeg)

print(f'Dataset EEG: N={N_eeg}, características={X_eeg.shape[1]}')
print(f'Clases: {[f"{s}(n={N_por_clase})" for s in nombres_estados]}')

## Parte 1 — El modelo generativo GMM

Un GMM con $K$ componentes modela la densidad como:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

donde $\pi_k \geq 0$, $\sum_k \pi_k = 1$ son los **pesos de mezcla**.

El modelo generativo es:
1. Muestrear componente: $z \sim \text{Categorical}(\boldsymbol{\pi})$
2. Muestrear observación: $\mathbf{x} \mid z=k \sim \mathcal{N}(\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$

La variable $z$ es **latente** (no observada) — esta es la clave que hace al EM necesario.

In [ ]:
# ── Visualización del GMM generativo ─────────────────────────────────────────
def dibujar_elipse_gaussiana(ax, media, cov, nivel=0.95, color='blue', lw=2, alpha=0.3):
    """Dibuja la elipse de confianza de una gaussiana 2D."""
    vals, vecs = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    chi2 = stats.chi2.ppf(nivel, df=2)
    w, h = 2 * np.sqrt(chi2 * vals)
    elipse = Ellipse(media, w, h, angle=ang, alpha=alpha,
                      facecolor=color, edgecolor=color, lw=lw)
    ax.add_patch(elipse)

# Proyección 2D: delta vs alpha (las más discriminativas)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_k = ['#3B82F6', '#F59E0B', '#10B981']

for k in range(3):
    mask = y_eeg == k
    axes[0].scatter(X_eeg[mask, 0], X_eeg[mask, 2],
                     c=colores_k[k], s=15, alpha=0.6, label=nombres_estados[k])
    dibujar_elipse_gaussiana(
        axes[0], medias_eeg[k][[0, 2]],
        covs_eeg[k][np.ix_([0, 2], [0, 2])],
        color=colores_k[k]
    )

axes[0].set(xlabel=nombres_bandas[0], ylabel=nombres_bandas[2],
            title='Dataset EEG — estados de vigilancia\nElipses de confianza 95%')
axes[0].legend(fontsize=8)

# Distribución marginal de Delta para cada estado
x_range = np.linspace(-1, 7, 300)
for k in range(3):
    pdf = stats.norm.pdf(x_range, medias_eeg[k, 0], np.sqrt(covs_eeg[k][0, 0]))
    axes[1].plot(x_range, pdf, color=colores_k[k], lw=2.5, label=nombres_estados[k])
    mask = y_eeg == k
    axes[1].hist(X_eeg[mask, 0], bins=25, density=True, alpha=0.3,
                  color=colores_k[k], edgecolor='none')

# Mezcla total (GMM)
mezcla_pdf = sum(
    (1/3) * stats.norm.pdf(x_range, medias_eeg[k, 0], np.sqrt(covs_eeg[k][0, 0]))
    for k in range(3)
)
axes[1].plot(x_range, mezcla_pdf, 'k--', lw=2, label='Mezcla total p(x)')
axes[1].set(xlabel=nombres_bandas[0], ylabel='Densidad',
            title='Distribución marginal de potencia delta\n'
                  'y mezcla de gaussianas resultante')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Parte 2 — Algoritmo EM desde cero

El EM maximiza la **ELBO** (Evidence Lower BOund) iterando:

**Paso E** (Expectation) — calcula las responsabilidades:
$$r_{ik} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j} \pi_j \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

**Paso M** (Maximization) — actualiza los parámetros:
$$N_k = \sum_i r_{ik}, \quad \boldsymbol{\mu}_k = \frac{1}{N_k}\sum_i r_{ik}\mathbf{x}_i, \quad
\boldsymbol{\Sigma}_k = \frac{1}{N_k}\sum_i r_{ik}(\mathbf{x}_i-\boldsymbol{\mu}_k)(\mathbf{x}_i-\boldsymbol{\mu}_k)^\top$$

In [ ]:
# ── EM para GMM desde cero (aritmética en log-espacio) ───────────────────────
def log_pdf_gaussiana(X, mu, Sigma):
    """Log-densidad gaussiana multivariada (estable numéricamente)."""
    d = X.shape[1]
    try:
        L    = np.linalg.cholesky(Sigma + 1e-6 * np.eye(d))
        diff = X - mu
        sol  = np.linalg.solve(L, diff.T)
        log_det = 2 * np.sum(np.log(np.diag(L)))
        maha    = np.sum(sol**2, axis=0)
        return -0.5 * (d * np.log(2*np.pi) + log_det + maha)
    except np.linalg.LinAlgError:
        return np.full(len(X), -np.inf)


def em_gmm(X, K, n_iter=100, tol=1e-4, rng_=None):
    """
    Ajusta un GMM con el algoritmo EM.
    Inicialización: k-means.
    Retorna: pesos, medias, covarianzas, log-verosimilitudes, responsabilidades.
    """
    rng_ = rng_ or np.random.default_rng()
    N, d = X.shape

    # Inicialización con k-means
    km = KMeans(n_clusters=K, random_state=int(rng_.integers(1000)), n_init=3)
    km.fit(X)
    mus   = km.cluster_centers_.copy()
    Sigs  = np.array([np.cov(X[km.labels_ == k].T) + 1e-4*np.eye(d)
                       if (km.labels_ == k).sum() > d else np.eye(d)
                       for k in range(K)])
    pis   = np.array([(km.labels_ == k).mean() for k in range(K)])
    pis  /= pis.sum()

    log_liks = []

    for it in range(n_iter):
        # ── Paso E: responsabilidades en log-espacio ─────────────────────────
        log_r = np.zeros((N, K))
        for k in range(K):
            log_r[:, k] = np.log(pis[k] + 1e-300) + log_pdf_gaussiana(X, mus[k], Sigs[k])

        log_norm  = logsumexp(log_r, axis=1, keepdims=True)
        log_r    -= log_norm
        r         = np.exp(log_r)             # responsabilidades (N x K)

        log_lik = log_norm.mean()
        log_liks.append(log_lik)

        # Criterio de convergencia
        if it > 0 and abs(log_liks[-1] - log_liks[-2]) < tol:
            break

        # ── Paso M: actualización de parámetros ──────────────────────────────
        Nk = r.sum(axis=0) + 1e-300
        pis = Nk / N
        for k in range(K):
            mus[k]  = (r[:, k:k+1] * X).sum(axis=0) / Nk[k]
            diff    = X - mus[k]
            Sigs[k] = (r[:, k:k+1] * diff).T @ diff / Nk[k] + 1e-4*np.eye(d)

    return pis, mus, Sigs, log_liks, r


# Ajustar GMM con K=3 (conocemos la verdad)
K_true = 3
pis_est, mus_est, Sigs_est, log_liks, r_est = em_gmm(
    X_eeg.astype(float), K_true, n_iter=200, rng_=rng
)

print(f'EM convergió en {len(log_liks)} iteraciones')
print(f'Log-verosimilitud final: {log_liks[-1]:.4f}')
print(f'Pesos estimados: {pis_est}')

# ── Curva de convergencia ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(log_liks, 'steelblue', lw=2.5)
ax.set(xlabel='Iteración EM', ylabel='Log-verosimilitud media',
       title='Convergencia del algoritmo EM para GMM con K=3\nDataset EEG — estados de vigilancia')
plt.tight_layout()
plt.show()

## Parte 3 — GMM vs k-means: asignaciones blandas vs duras

k-means asigna cada punto a exactamente un clúster (**asignación dura**).
El GMM asigna probabilidades de pertenencia a cada componente (**asignación blanda**).

La diferencia es especialmente visible en puntos ambiguos en la región de transición entre clases.

In [ ]:
# ── GMM vs k-means: comparación visual de asignaciones ───────────────────────
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
km3.fit(X_eeg)
asig_km  = km3.labels_
asig_gmm = r_est.argmax(axis=1)    # asignación dura del GMM (argmax)
incert_gmm = 1 - r_est.max(axis=1) # incertidumbre = 1 - prob máxima

# Remapear etiquetas GMM para coincidir con la verdad
from scipy.optimize import linear_sum_assignment
def remap_labels(pred, true, K):
    cost = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            cost[i, j] = np.sum((pred == i) & (true == j))
    row, col = linear_sum_assignment(-cost)
    new_pred = np.zeros_like(pred)
    for r, c in zip(row, col):
        new_pred[pred == r] = c
    return new_pred

asig_gmm_rm = remap_labels(asig_gmm, y_eeg, 3)
asig_km_rm  = remap_labels(asig_km,  y_eeg, 3)
acc_gmm = (asig_gmm_rm == y_eeg).mean()
acc_km  = (asig_km_rm  == y_eeg).mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Verdad
for k in range(3):
    mask = y_eeg == k
    axes[0].scatter(X_eeg[mask, 0], X_eeg[mask, 2],
                     c=colores_k[k], s=15, alpha=0.7, label=nombres_estados[k])
axes[0].set(xlabel=nombres_bandas[0], ylabel=nombres_bandas[2],
            title='Verdad (etiquetas reales)')
axes[0].legend(fontsize=7)

# k-means
for k in range(3):
    mask = asig_km_rm == k
    axes[1].scatter(X_eeg[mask, 0], X_eeg[mask, 2],
                     c=colores_k[k], s=15, alpha=0.7)
axes[1].set(xlabel=nombres_bandas[0], ylabel=nombres_bandas[2],
            title=f'k-means (exactitud={acc_km:.3f})\nAsignación dura')

# GMM con incertidumbre
for k in range(3):
    mask = asig_gmm_rm == k
    axes[2].scatter(X_eeg[mask, 0], X_eeg[mask, 2],
                     c=colores_k[k], s=15,
                     alpha=np.clip(1 - incert_gmm[mask]*2, 0.2, 1.0))

# Puntos con alta incertidumbre
alta_incert = incert_gmm > 0.3
axes[2].scatter(X_eeg[alta_incert, 0], X_eeg[alta_incert, 2],
                 c='none', edgecolors='black', s=40, lw=1.5,
                 label=f'Incertidumbre >0.3 (n={alta_incert.sum()})')
axes[2].set(xlabel=nombres_bandas[0], ylabel=nombres_bandas[2],
            title=f'GMM (exactitud={acc_gmm:.3f})\nAlfa=confianza de asignación')
axes[2].legend(fontsize=8)

plt.suptitle('GMM vs k-means — asignaciones blandas vs duras\nDataset EEG estados de vigilancia',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print(f'Exactitud k-means: {acc_km:.3f} | Exactitud GMM: {acc_gmm:.3f}')
print(f'Puntos con incertidumbre GMM > 30%: {alta_incert.sum()} ({alta_incert.mean():.1%})')

## Parte 4 — Selección de K: BIC, AIC y silueta

¿Cómo elegir el número de componentes $K$ sin saber la verdad?

| Criterio | Fórmula | Penaliza |
|---|---|---|
| **BIC** | $-2\log\hat{L} + p\log N$ | Fuertemente la complejidad (preferible con N grande) |
| **AIC** | $-2\log\hat{L} + 2p$ | Menos que BIC (tiende a sobreajustar K) |
| **Silueta** | Media de $(b_i - a_i)/\max(a_i, b_i)$ | No asume forma; más lento |

In [ ]:
# ── Selección de K con BIC, AIC y silueta ────────────────────────────────────
Ks = range(1, 9)
bics, aics, silhouettes = [], [], []

for K in Ks:
    gmm_k = GaussianMixture(n_components=K, covariance_type='full',
                              n_init=5, random_state=42)
    gmm_k.fit(X_eeg)
    bics.append(gmm_k.bic(X_eeg))
    aics.append(gmm_k.aic(X_eeg))
    if K >= 2:
        labels = gmm_k.predict(X_eeg)
        sil = silhouette_score(X_eeg, labels, sample_size=500, random_state=42)
    else:
        sil = np.nan
    silhouettes.append(sil)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(Ks), bics, 'steelblue', lw=2.5, marker='o', ms=7, label='BIC')
axes[0].plot(list(Ks), aics, 'tomato',    lw=2.5, marker='s', ms=7, label='AIC')
k_bic = list(Ks)[np.argmin(bics)]
k_aic = list(Ks)[np.argmin(aics)]
axes[0].axvline(k_bic, color='steelblue', ls='--', lw=1.5, label=f'BIC óptimo K={k_bic}')
axes[0].axvline(k_aic, color='tomato',    ls='--', lw=1.5, label=f'AIC óptimo K={k_aic}')
axes[0].set(xlabel='Número de componentes K', ylabel='Criterio de información',
            title='Selección de K — BIC y AIC\n(menor = mejor)')
axes[0].legend(fontsize=9)

axes[1].plot(list(Ks)[1:], silhouettes[1:], 'seagreen', lw=2.5, marker='D', ms=7)
k_sil = list(Ks)[1:][np.argmax(silhouettes[1:])]
axes[1].axvline(k_sil, color='seagreen', ls='--', lw=1.5, label=f'Silueta óptima K={k_sil}')
axes[1].set(xlabel='Número de componentes K', ylabel='Índice de silueta (↑ mejor)',
            title='Selección de K — Índice de silueta')
axes[1].legend(fontsize=9)

plt.suptitle('Selección de número de componentes K para GMM\nDataset EEG estados de vigilancia',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print(f'BIC óptimo: K={k_bic} | AIC óptimo: K={k_aic} | Silueta óptima: K={k_sil}')
print(f'Verdad: K={K_true} — ¿Los criterios la recuperan?')

## Parte 5 — GMM para detección de anomalías en ECG

El GMM estima la densidad de los datos normales. Un punto con densidad
muy baja respecto al umbral aprendido se clasifica como anomalía.

Aplicamos esto para detectar latidos de ECG anómalos (arritmias) en un
fondo de latidos normales.

In [ ]:
# ── Detección de anomalías ECG con GMM ────────────────────────────────────────
# Dataset: latidos ECG — 5 características morfológicas
# Inspirado en MIT-BIH Arrhythmia Database
# Moody, G.B. & Mark, R.G. (2001). The impact of the MIT-BIH arrhythmia database.
# IEEE Engineering in Medicine and Biology, 20(3), 45–50.
# https://doi.org/10.1109/51.932724

nombres_feats_ecg = ['Amp R (mV)', 'Dur QRS (ms)', 'Área QRS', 'Intervalo RR (ms)', 'Eje (°)']

n_norm = 800
n_anom = 40

# Latidos normales: clusters ligeramente no gaussianos
X_norm_1 = rng.multivariate_normal([1.0, 80, 0.6, 800, 60],
                                     np.diag([0.04, 100, 0.01, 2500, 100]),
                                     n_norm // 2)
X_norm_2 = rng.multivariate_normal([0.9, 85, 0.55, 820, 65],
                                     np.diag([0.03, 80,  0.01, 2000, 80]),
                                     n_norm - n_norm // 2)
X_normal = np.vstack([X_norm_1, X_norm_2]).astype(np.float32)

# Latidos anómalos: morfología distinta (QRS ancho, eje desviado)
X_anomalo = np.column_stack([
    rng.normal(1.5, 0.3, n_anom),    # Amp R elevada
    rng.normal(140, 30,  n_anom),    # QRS ancho (BCRI/BCRD)
    rng.normal(1.2, 0.2, n_anom),    # Área QRS grande
    rng.normal(650, 80,  n_anom),    # RR corto (taquicardia)
    rng.normal(-20, 40,  n_anom),    # Eje muy desviado
]).astype(np.float32)

# Entrenar GMM solo con datos normales
gmm_ecg = GaussianMixture(n_components=2, covariance_type='full',
                            n_init=10, random_state=42)
gmm_ecg.fit(X_normal)

# Umbral: percentil 5 de la log-densidad en datos normales
log_dens_norm = gmm_ecg.score_samples(X_normal)
umbral        = np.percentile(log_dens_norm, 5)

# Evaluar en todos los datos (normal + anómalos)
X_todo  = np.vstack([X_normal, X_anomalo])
y_todo  = np.concatenate([np.zeros(n_norm), np.ones(n_anom)]).astype(int)
log_den = gmm_ecg.score_samples(X_todo)
y_pred  = (log_den < umbral).astype(int)

from sklearn.metrics import roc_auc_score, classification_report
auroc_anom = roc_auc_score(y_todo, -log_den)  # negativo: score anómalo = log_dens bajo

print('Detección de anomalías ECG con GMM:')
print(f'AUROC = {auroc_anom:.3f}')
print(f'Umbral: {umbral:.3f}')
print()
print(classification_report(y_todo, y_pred, target_names=['Normal', 'Anómalo']))

# Visualización: log-densidad por clase
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(log_dens_norm, bins=40, color='steelblue', alpha=0.7,
              density=True, label='Normales (train)')
axes[0].hist(gmm_ecg.score_samples(X_anomalo), bins=20, color='tomato',
              alpha=0.7, density=True, label='Anómalos')
axes[0].axvline(umbral, color='black', lw=2, ls='--',
                 label=f'Umbral (p5) = {umbral:.2f}')
axes[0].set(xlabel='Log-densidad GMM', ylabel='Densidad normalizada',
            title='Distribución de log-densidades\nLatidos normales vs anómalos')
axes[0].legend(fontsize=9)

# Curva ROC
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_todo, -log_den)
axes[1].plot(fpr, tpr, 'steelblue', lw=2.5, label=f'GMM (AUROC={auroc_anom:.3f})')
axes[1].plot([0,1],[0,1], 'gray', ls='--', lw=1)
axes[1].set(xlabel='FPR (1−Especificidad)', ylabel='TPR (Sensibilidad)',
            title='Curva ROC — Detección de anomalías ECG\nGMM como estimador de densidad',
            xlim=[-0.02,1.02], ylim=[-0.02,1.02])
axes[1].set_aspect('equal')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Parte 6 — Naive Bayes gaussiano, LDA y QDA como casos especiales del GMM

Todos los clasificadores gaussianos son casos especiales de un GMM con etiquetas conocidas:

| Modelo | Covarianza $\Sigma_k$ | Frontera de decisión |
|---|---|---|
| **Naive Bayes gaussiano** | Diagonal, distinta por clase | Cuadrática |
| **LDA** | Compartida entre clases: $\Sigma_k = \Sigma$ | Lineal |
| **QDA** | Distinta por clase (sin restricción) | Cuadrática |
| **GMM supervisionado** | Igual que QDA | Cuadrática |

La diferencia está en cuántos parámetros se estiman y cómo se regulariza la covarianza.

In [ ]:
# ── Comparación de clasificadores gaussianos en el dataset EEG ────────────────
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

clasificadores = [
    ('Naive Bayes gaussiano', GaussianNB()),
    ('LDA',                   LinearDiscriminantAnalysis()),
    ('QDA',                   QuadraticDiscriminantAnalysis()),
    ('GMM supervisionado',    GaussianMixture(n_components=3, covariance_type='full',
                                              n_init=5, random_state=42)),
]

print(f'{"Modelo":<25}  {"Exactitud CV":>12}  {"± std":>7}  {"Nº parámetros"}')
print('─' * 70)

resultados_gauss = []
for nombre, clf in clasificadores[:-1]:  # GMM no es un clasificador sklearn estándar
    scores = cross_val_score(clf, X_eeg, y_eeg, cv=cv5, scoring='accuracy')
    resultados_gauss.append((nombre, scores.mean(), scores.std()))
    print(f'{nombre:<25}  {scores.mean():12.3f}  {scores.std():7.3f}')

# GMM supervisionado (ajustar una por clase y clasificar por Bayes)
def gmm_clf_supervisionado(X_tr, y_tr, X_te, K_por_clase=1):
    clases = np.unique(y_tr)
    modelos = {}
    prevs   = {}
    for k in clases:
        gm = GaussianMixture(n_components=K_por_clase, covariance_type='full',
                              n_init=5, random_state=42)
        gm.fit(X_tr[y_tr == k])
        modelos[k] = gm
        prevs[k]   = np.mean(y_tr == k)
    # Predict
    log_posts = np.zeros((len(X_te), len(clases)))
    for k in clases:
        log_posts[:, k] = (modelos[k].score_samples(X_te) +
                            np.log(prevs[k] + 1e-300))
    return log_posts.argmax(axis=1)

# CV manual para GMM supervisionado
acc_gmm_sup = []
for tr_idx, te_idx in cv5.split(X_eeg, y_eeg):
    preds = gmm_clf_supervisionado(X_eeg[tr_idx], y_eeg[tr_idx], X_eeg[te_idx])
    acc_gmm_sup.append((preds == y_eeg[te_idx]).mean())

resultados_gauss.append(('GMM supervisionado', np.mean(acc_gmm_sup), np.std(acc_gmm_sup)))
print(f'{"GMM supervisionado":<25}  {np.mean(acc_gmm_sup):12.3f}  {np.std(acc_gmm_sup):7.3f}')

# Visualización
fig, ax = plt.subplots(figsize=(9, 4))
nombres_g = [r[0] for r in resultados_gauss]
medias_g  = [r[1] for r in resultados_gauss]
stds_g    = [r[2] for r in resultados_gauss]
colores_g = ['#3B82F6','#F59E0B','#EF4444','#10B981']
ax.barh(nombres_g, medias_g, xerr=stds_g, color=colores_g, alpha=0.8,
         capsize=5)
ax.axvline(1/3, color='gray', ls='--', lw=1, label='Azar (0.333)')
ax.set(xlabel='Exactitud (5-fold CV)', title='Clasificadores gaussianos — Dataset EEG\n'
               'Todos son casos especiales del GMM con etiquetas observadas')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## ✏️ Ejercicios

Los ejercicios usan datos de **vigilancia EEG** de sujetos sanos en reposo.
Distribución simulada a partir de las estadísticas de potencia espectral
reportadas en Berry et al. (2015) AASM Scoring Manual.

1. **Sensibilidad a la inicialización.** Ajusta el GMM con K=3 en el dataset EEG
   iniciando con k-means, aleatorio y con `init_params='random_from_data'`.
   ¿Las log-verosimilitudes finales difieren? ¿Las responsabilidades son similares?
   Repite 20 veces con distintas semillas y reporta la varianza de la log-verosimilitud.

2. **Tipos de covarianza.** Compara los cuatro tipos de covarianza de `GaussianMixture`:
   `full`, `tied`, `diag`, `spherical`. Para cada uno, reporta BIC, AIC y exactitud
   (etiquetas conocidas). ¿Cuál es el mejor compromiso complejidad/rendimiento?

3. **GMM para datos faltantes.** Simula un 20% de datos faltantes en el dataset EEG
   (NaN aleatorios). Implementa un pipeline de imputación iterativa usando el GMM:
   (a) imputa con la media de la clase más probable, (b) reajusta el GMM,
   (c) repite hasta convergencia. Compara la exactitud con imputación simple por media.

4. **GMM de ventana deslizante para sueño.** Genera una señal EEG simulada de 8 horas
   con transiciones entre estados (ciclos de ~90 min). Ajusta un GMM global.
   Aplica el modelo con ventana de 30 s para segmentar el hipnograma.
   Compara con el hipnograma verdadero.

5. *(Desafío)* **GMM en datos ISRUC reales.** Descarga el dataset ISRUC-Sleep
   (https://sleeptight.isr.uc.pt/). Extrae características de potencia espectral
   por banda. Ajusta un GMM con K=5 (AASM: W, N1, N2, N3, REM). Compara con las
   etiquetas del experto usando la puntuación de información mutua normalizada (NMI).

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| EEG vigilancia (simulado) | Berry, R.B. et al. (2015). *AASM Scoring Manual* v2.2. https://aasm.org/clinical-resources/scoring-manual/ | Dataset principal S10 |
| MIT-BIH Arrhythmia | Moody, G.B. & Mark, R.G. (2001). *IEEE Eng. Med. Biol.* 20(3):45–50. https://doi.org/10.1109/51.932724 | Ejercicio de detección de anomalías |
| ISRUC-Sleep | Khalighi, S. et al. (2016). *Computers Methods Programs Biomed.* 124:180–192. https://sleeptight.isr.uc.pt/ | Ejercicio desafío S10 |